In [ ]:
import pandas as pd
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Data Preprocessing

First, we need to clean up the raw data. In this step, we will:
1. **Drop irrelevant identifiers:** `user_id`, `ad_id`, and `interaction_timestamps` don't help our models.
2. **Keep TF-IDF:** We are leaving the `tfidf` columns untouched because they represent the actual text content of the ads, which is crucial for predicting engagement.
3. **Yeo-Johnson Transformation:** Because the distribution of `clicks` is exponential, we apply Yeo-Johnson transformation to make the distribution more normal. This will allow us to get better results for our SVM model. The Yeo-Johnson distribution is good for exponential distributions and works with 0 and negative values.
4. **Normalization:** Apply Min-Max scaling to the continuous features.
5. **One-Hot Encoding:** Because the numbers assigned to the categorical features are arbitrary, we use one-hot encoding to prevent bias in our models.

In [ ]:
# 1. Load the original dataset
df = pd.read_csv('ad_campaign_data.csv')

# 2. Drop unnecessary identifiers and timestamps
columns_to_drop = ['user_id', 'ad_id', 'interaction_timestamps']
df_processed = df.drop(columns=columns_to_drop)

# 3. Yeo-Johnson transformation for SVMs
df_transformed = df.copy()
# Drop columns we're not using
df_transformed.drop(columns=columns_to_drop, inplace=True)
# Apply Yeo-Johnson transformation to clicks to reduce skew
transformer = PowerTransformer(method='yeo-johnson')
df_transformed['clicks'] = transformer.fit_transform(df_transformed[['clicks']])
# Visualize change in distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Clicks before transformation
axes[0].hist(df_processed['clicks'], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Clicks (Raw)")
axes[0].set_xlabel("Clicks")
# Clicks after transformation
axes[1].hist(df_transformed['clicks'], bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("Clicks (Yeo-Johnson Transformed)")
axes[1].set_xlabel("Clicks (Transformed)")
plt.tight_layout()
plt.show()

# 4. Normalization
continuous_features = ['age', 'impressions', 'clicks', 'previous_interaction_score', 'sentiment_score']
scaler = MinMaxScaler()
df_processed[continuous_features] = scaler.fit_transform(df_processed[continuous_features])
df_transformed[continuous_features] = scaler.fit_transform(df_transformed[continuous_features]) # for SVMs

# 5. One-Hot Encoding
encode_vars = ['gender', 'location', 'device_type', 'ad_category']
df_encoded = pd.get_dummies(df_processed, columns=encode_vars, prefix=encode_vars, dtype=int)
# For SVM model (with transformed clicks)
df_encoded_svm = pd.get_dummies(df_transformed, columns=encode_vars, prefix=encode_vars, dtype=int)

# 6. Move the target variable ('conversions') to the end
encoded_feature_cols = [col for col in df_encoded.columns if col != 'conversions']
df_encoded = df_encoded[encoded_feature_cols + ['conversions']]
df_encoded_svm = df_encoded_svm[encoded_feature_cols + ['conversions']]

# 7. Save to a clean CSV for the team to use
output_filename = 'final_project_data.csv'
df_encoded.to_csv(output_filename, index=False)
df_encoded_svm.to_csv('svm_project_data.csv', index=False)

print(f"Final dataset shape: {df_encoded.shape}")
df_encoded.head()

## Exploratory Data Analysis (EDA)
Let's visualize the distributions of our continuous and categorical features to understand the shape of our data before feeding it into our models.

In [ ]:
# Set up the visual style
sns.set_theme(style="whitegrid")

# Define our feature groups
continuous_features = ['age', 'impressions', 'clicks', 'engagement_duration', 'previous_interaction_score', 'sentiment_score']
categorical_features = ['gender', 'location', 'device_type', 'ad_category', 'conversions']

# --- Plot 1: Continuous Distributions ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions of Continuous Features', fontsize=16)
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.histplot(df[col], kde=True, ax=axes[i], bins=30, color='skyblue')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

# --- Plot 2: Categorical Distributions ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions of Categorical Features', fontsize=16)
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    val_counts = df[col].value_counts().sort_index()
    sns.barplot(x=val_counts.index, y=val_counts.values, hue=val_counts.index, ax=axes[i], palette='Set2', legend=False)
    axes[i].set_title(col)
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')

axes[5].set_visible(False) # Hide the last empty subplot
plt.tight_layout()
plt.show()

# --- Plot 3: Correlation Matrix for Continuous Features ---
continuous_df = df_processed[continuous_features] # dataframe with continuous features
sns.heatmap(continuous_df.corr(), annot=True, vmin=-1, vmax=1, center=0, cmap='coolwarm', linewidths=0.5, linecolor='black', square=True)
plt.title('Correlation Matrix for Continuous Features', fontsize=16)
plt.show()

# --- Plot 4: Pairplot ---
pairplot_features = continuous_features
pairplot_features.append('conversions')
pairplot_df = df_processed[pairplot_features]
sns.set_style('whitegrid')
sns.pairplot(pairplot_df, hue='conversions')
plt.show()

## Hidden Layer Models 3-4

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, f1_score


In [ ]:
def create_mlp_model(num_hidden_layers=2, learning_rate=0.01, momentum=0.0,
                     l2_lambda=0.0, dropout_rate=0.0,
                     num_hidden_units_1=128, num_hidden_units_2=64, num_hidden_units_3=32, num_hidden_units_4=16):
    model = Sequential()

    model.add(Dense(num_hidden_units_1, activation='relu', input_shape=(num_features,),
                    kernel_regularizer=l2(l2_lambda)))
    model.add(Dropout(dropout_rate))

    model.add(Dense(num_hidden_units_2, activation='relu', kernel_regularizer=l2(l2_lambda)))
    model.add(Dropout(dropout_rate))

    if num_hidden_layers >= 3:
        model.add(Dense(num_hidden_units_3, activation='relu', kernel_regularizer=l2(l2_lambda)))
        model.add(Dropout(dropout_rate))

    if num_hidden_layers == 4:
        model.add(Dense(num_hidden_units_4, activation='relu', kernel_regularizer=l2(l2_lambda)))
        model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = SGD(learning_rate=learning_rate, momentum=momentum)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
class_weights = class_weight.compute_class_weight('balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train.ravel())

class_weights_dict = dict(enumerate(class_weights))

print("Calculated Class Weights:", class_weights_dict)

In [ ]:
final_params_3_layer = {
    'model__num_hidden_units_3': 16,
    'model__num_hidden_units_2': 32,
    'model__num_hidden_units_1': 64,
    'model__num_hidden_layers': 3,
    'model__momentum': 0.0,
    'model__learning_rate': 0.001,
    'model__l2_lambda': 0.0,
    'model__dropout_rate': 0.1,
    'epochs': 200,
    'batch_size': 64
}

print("\n--- Training Final 3-Hidden Layer Weighted MLP Model ---")

final_mlp_3_layer_weighted = KerasClassifier(model=create_mlp_model,
                                             num_hidden_layers=final_params_3_layer['model__num_hidden_layers'],
                                             learning_rate=final_params_3_layer['model__learning_rate'],
                                             momentum=final_params_3_layer['model__momentum'],
                                             l2_lambda=final_params_3_layer['model__l2_lambda'],
                                             dropout_rate=final_params_3_layer['model__dropout_rate'],
                                             num_hidden_units_1=final_params_3_layer['model__num_hidden_units_1'],
                                             num_hidden_units_2=final_params_3_layer['model__num_hidden_units_2'],
                                             num_hidden_units_3=final_params_3_layer['model__num_hidden_units_3'],
                                             epochs=final_params_3_layer['epochs'],
                                             batch_size=final_params_3_layer['batch_size'],
                                             verbose=0)

final_mlp_3_layer_weighted.fit(X_train, y_train.ravel(), class_weight=class_weights_dict)

final_model_3_layer_path = 'final_mlp_3_layer_weighted.keras'
final_mlp_3_layer_weighted.model_.save(final_model_3_layer_path)
print(f"Final 3-Hidden Layer Weighted MLP Model saved to: {final_model_3_layer_path}")

In [ ]:
weights_path_3_layer = 'final_mlp_3_layer_weighted_weights.weights.h5'
final_mlp_3_layer_weighted.model_.save_weights(weights_path_3_layer)
print(f"Final 3-Hidden Layer Weighted MLP Model weights saved to: {weights_path_3_layer}")

In [ ]:
final_params_4_layer = {
    'model__num_hidden_units_3': 16,
    'model__num_hidden_units_2': 32,
    'model__num_hidden_units_1': 64,
    'model__num_hidden_layers': 4,
    'model__num_hidden_units_4': 8,
    'model__momentum': 0.0,
    'model__learning_rate': 0.001,
    'model__l2_lambda': 0.0,
    'model__dropout_rate': 0.1,
    'epochs': 200,
    'batch_size': 64
}

print("\n--- Training Final 4-Hidden Layer Weighted MLP Model ---")

final_mlp_4_layer_weighted = KerasClassifier(model=create_mlp_model,
                                             num_hidden_layers=final_params_4_layer['model__num_hidden_layers'],
                                             learning_rate=final_params_4_layer['model__learning_rate'],
                                             momentum=final_params_4_layer['model__momentum'],
                                             l2_lambda=final_params_4_layer['model__l2_lambda'],
                                             dropout_rate=final_params_4_layer['model__dropout_rate'],
                                             num_hidden_units_1=final_params_4_layer['model__num_hidden_units_1'],
                                             num_hidden_units_2=final_params_4_layer['model__num_hidden_units_2'],
                                             num_hidden_units_3=final_params_4_layer['model__num_hidden_units_3'],
                                             num_hidden_units_4=final_params_4_layer['model__num_hidden_units_4'],
                                             epochs=final_params_4_layer['epochs'],
                                             batch_size=final_params_4_layer['batch_size'],
                                             verbose=0)

final_mlp_4_layer_weighted.fit(X_train, y_train.ravel(), class_weight=class_weights_dict)

final_model_4_layer_path = 'final_mlp_4_layer_weighted.keras'
final_mlp_4_layer_weighted.model_.save(final_model_4_layer_path)
print(f"Final 4-Hidden Layer Weighted MLP Model saved to: {final_model_4_layer_path}")

In [ ]:
weights_path_4_layer = 'final_mlp_4_layer_weighted_weights.weights.h5'
final_mlp_4_layer_weighted.model_.save_weights(weights_path_4_layer)
print(f"Final 4-Hidden Layer Weighted MLP Model weights saved to: {weights_path_4_layer}")

In [ ]:
from tensorflow.keras.models import load_model

loaded_mlp_3_layer = load_model('final_mlp_3_layer_weighted.keras', compile=False)
print("Loaded 3-Hidden Layer Model Summary:")
loaded_mlp_3_layer.summary()

loaded_scikeras_3_layer = KerasClassifier(model=create_mlp_model, model__num_hidden_layers=3,
                                          model__learning_rate=0.001, model__momentum=0.0,
                                          model__l2_lambda=0.0, model__dropout_rate=0.1,
                                          model__num_hidden_units_1=64, model__num_hidden_units_2=32,
                                          model__num_hidden_units_3=16, epochs=200, batch_size=64)
loaded_scikeras_3_layer.initialize(X=X_test, y=y_test)
loaded_scikeras_3_layer.set_params(model=loaded_mlp_3_layer)

loaded_mlp_4_layer = load_model('final_mlp_4_layer_weighted.keras', compile=False)
print("\nLoaded 4-Hidden Layer Model Summary:")
loaded_mlp_4_layer.summary()

loaded_scikeras_4_layer = KerasClassifier(model=create_mlp_model, model__num_hidden_layers=4,
                                          model__learning_rate=0.001, model__momentum=0.0,
                                          model__l2_lambda=0.0, dropout_rate=0.1,
                                          model__num_hidden_units_1=64, model__num_hidden_units_2=32,
                                          model__num_hidden_units_3=16, model__num_hidden_units_4=8,
                                          epochs=200, batch_size=64)
loaded_scikeras_4_layer.initialize(X=X_test, y=y_test)
loaded_scikeras_4_layer.set_params(model=loaded_mlp_4_layer)

print("\nModels loaded successfully. You can now use `loaded_scikeras_3_layer` and `loaded_scikeras_4_layer` for predictions.")